# Experiment 6: Classification of Credit Card Default Risk using SVM

**Aim:** To implement Support Vector Machine (SVM) classifiers with different kernel functions and compare their performance for predicting credit card default.

**Roll No:** 58 | **Batch:** SAA3

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [ ]:
# Load credit card default dataset
df = pd.read_csv('datasets/credit.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

## 3. Data Exploration

In [ ]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Dataset info
print("\nDataset Info:")
df.info()

# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check target distribution
if 'default' in df.columns:
    target_col = 'default'
elif 'Default' in df.columns:
    target_col = 'Default'
else:
    # Assume last column is target
    target_col = df.columns[-1]

print(f"Target column: {target_col}")
print(f"\nClass distribution:\n{df[target_col].value_counts()}")

# Visualize target distribution
plt.figure(figsize=(6, 4))
df[target_col].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Credit Card Default Distribution')
plt.xlabel('Default Status (0=No, 1=Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 4. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Handle categorical columns if any
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Feature scaling (important for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaling completed")

## 5. SVM Model Training with Different Kernels

In [ ]:
# Define kernels to test
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = {}

for kernel in kernels:
    print(f"\n{'='*60}")
    print(f"Training SVM with {kernel.upper()} kernel")
    print('='*60)
    
    # Create and train SVM model
    svm_model = SVC(kernel=kernel, random_state=42)
    svm_model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = svm_model.predict(X_test_scaled)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results[kernel] = {
        'model': svm_model,
        'predictions': y_pred,
        'accuracy': accuracy
    }
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix - {kernel.upper()} Kernel')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

## 6. Model Comparison

In [ ]:
# Compare accuracies
accuracies = {kernel: results[kernel]['accuracy'] for kernel in kernels}

print("\nModel Comparison:")
print("="*40)
for kernel, acc in accuracies.items():
    print(f"{kernel.upper():<10} : {acc:.4f}")

# Find best model
best_kernel = max(accuracies, key=accuracies.get)
print(f"\nBest performing kernel: {best_kernel.upper()} with accuracy {accuracies[best_kernel]:.4f}")

In [ ]:
# Visualize accuracy comparison
plt.figure(figsize=(10, 6))
kernels_list = list(accuracies.keys())
accuracies_list = list(accuracies.values())

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = plt.bar(kernels_list, accuracies_list, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies_list):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.4f}',
             ha='center', va='bottom', fontweight='bold')

plt.title('SVM Kernel Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Kernel Type', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## 7. Conclusion

We successfully implemented SVM classifiers with four different kernels (Linear, RBF, Polynomial, and Sigmoid) to predict credit card default risk. Each kernel showed varying performance levels, with the best model achieving strong classification accuracy. The RBF and Linear kernels typically perform well for this type of binary classification task. Feature scaling was crucial for SVM performance, and the confusion matrices helped us understand model predictions in detail.